# Tests: `fastermodels.measurement` (source `nbs/04_measurement.ipynb`)

In [ ]:
from fastcore.test import *
import json, re, tempfile
from pathlib import Path

import numpy as np

import fastermodels
from fastermodels import measurement

In [ ]:
_ENGINE = 'ab' * 32
_LAT = dict(schema_version=1, kind='latency', run_id='l1', measured_at='2026-09-28T10:00:00Z', series='2026-09-28',
            harness_commit='deadbee', fasterai_version='0.4.0', report_url=None,
            architecture='resnet50', graph_hash='cd' * 32, input_resolution=224, macs=2_900_000_000,
            params=17_000_000, variant='p30-int8', recipe={'prune': 0.30, 'quant': 'int8', 'calib_n': 512},
            weights='trained', model_id='org/resnet50-p30', model_revision='a1b2c3d',
            precision='int8', int8_mode='explicit', engine_hash=_ENGINE, size_mb=18.2, engine_layers=61,
            reformat_layers=2, layer_precisions={'int8': 52, 'fp16': 9},
            device='jetson-nano-4gb', device_host='own', runtime='tensorrt', runtime_version='8.2.1',
            platform_version='jetpack-4.6.1', power_mode='MAXN', clocks_locked=True, batch_size=4,
            temp_start_c=41.0, temp_end_c=47.5, gpu_freq_mhz=921.6, throttled=False, warmup_s=10.0,
            duration_s=60.0, repeats=5, latency_spread_pct=1.2,
            latency_ms_p50=8.0, latency_ms_p90=8.4, latency_ms_p99=9.1, latency_scope='engine execute',
            host_latency_ms=8.5, latency_source='measured')

# the same graph with random weights: no model behind it, no accuracy, a first-class latency row
_RAND = {**{k: v for k, v in _LAT.items() if k not in ('model_id', 'model_revision')},
         'run_id': 'l2', 'weights': 'random', 'engine_hash': 'ef' * 32}

_QUAL = dict(schema_version=1, kind='quality', run_id='q1', measured_at='2026-09-28T11:00:00Z', series='2026-09-28',
             harness_commit='deadbee', fasterai_version='0.4.0', report_url=None,
             model_id='org/resnet50-p30', model_revision='a1b2c3d', model_hash=None, weights_license='apache-2.0',
             engine_hash=_ENGINE, precision='int8', device='jetson-nano-4gb',
             metric='top1', eval_set='imagenet-val', eval_set_hash='sha256:0f', dataset_license='custom',
             eval_n=50_000, accuracy=78.9, ci_low=78.5, ci_high=79.3, ci_method='wilson',
             measured_through='deployed_engine', argmax_parity=99.7, parity_floor=99.0, baseline_run_id='q0')


def edit(row, **change):
    "`row` with some fields changed; a value of `...` removes the field"
    return {k: v for k, v in {**row, **change}.items() if v is not ...}


def problems(row, **change): return measurement.validate(edit(row, **change))
def names(ps): return {p.split(':')[0] for p in ps}

In [ ]:
test_eq(set(_LAT), set(measurement.FIELDS['latency']))
test_eq(set(_QUAL), set(measurement.FIELDS['quality']))
test_eq(measurement.SCHEMA_VERSION, 1)
for _r in (_LAT, _RAND, _QUAL): test_eq(measurement.validate(_r), [])

# the shared fields are defined once, and the schema is data
_shared = ('schema_version', 'kind', 'run_id', 'measured_at', 'series', 'harness_commit', 'fasterai_version', 'report_url')
for _f in _shared: test_is(measurement.FIELDS['latency'][_f], measurement.FIELDS['quality'][_f])

# a latency row needs no accuracy, a quality row no latency
test_eq(problems(_LAT, accuracy=78.9), ['accuracy: unknown field for a latency row'])
test_eq(problems(_QUAL, latency_ms_p50=8.0), ['latency_ms_p50: unknown field for a quality row'])

# optional fields may be absent: absent means not recorded
test_eq(problems(_LAT, series=..., fasterai_version=..., platform_version=..., temp_start_c=..., latency_ms_p90=...,
                 latency_ms_p99=..., engine_layers=..., reformat_layers=..., layer_precisions=..., model_id=...,
                 model_revision=...), [])
test_eq(problems(_QUAL, model_hash=..., device=..., baseline_run_id=..., argmax_parity=...), [])
# JSON writes 8 as 8: an int is accepted where a float is expected
test_eq(problems(_LAT, latency_ms_p50=8), [])

# numpy scalars, as a harness produces them, are numbers; a numpy or Python bool is not
test_eq(problems(_LAT, batch_size=np.int64(4), macs=np.uint64(2_900_000_000), latency_ms_p50=np.float64(8.0)), [])
test_eq(problems(_QUAL, eval_n=np.int32(50_000), accuracy=np.float32(78.9)), [])
test_eq(names(problems(_LAT, batch_size=np.bool_(True))), {'batch_size'})
test_eq(names(problems(_LAT, latency_ms_p50=np.float64('nan'))), {'latency_ms_p50'})

# the module is reached by its name, its generic function names are not at the package top level
test_eq(fastermodels.measurement is measurement, True)
test_eq(hasattr(fastermodels, 'status') or hasattr(fastermodels, 'key'), False)

In [ ]:
# the kind decides the schema: missing or unknown, one clear problem and nothing else
test_eq(problems(_LAT, kind=...), ['kind: missing'])
test_eq(problems(_LAT, kind=None), ['kind: missing'])
test_eq(problems(_LAT, kind='speed'), ["kind: 'speed' not one of ('latency', 'quality')"])
test_eq(problems(_LAT, kind=['latency']), ["kind: ['latency'] not one of ('latency', 'quality')"])
# a row of one kind is not a row of the other
test_eq(names(problems(_LAT, kind='quality')) >= {'architecture', 'accuracy', 'weights_license'}, True)
# not a row at all
test_eq(measurement.validate([_LAT]), ['row: expected a dict, got list'])
test_eq(measurement.validate(None), ['row: expected a dict, got NoneType'])

In [ ]:
# missing required, by name
test_eq(problems(_LAT, run_id=...), ['run_id: missing'])
test_eq(problems(_LAT, latency_ms_p50=None), ['latency_ms_p50: missing'])
test_eq(problems(_LAT, graph_hash=...), ['graph_hash: missing'])
test_eq(problems(_LAT, weights=...), ['weights: missing'])
test_eq(problems(_QUAL, weights_license=...), ['weights_license: missing'])
test_eq(names(problems(_QUAL, model_id=..., eval_set_hash=...)), {'model_id', 'eval_set_hash'})

# wrong type
test_eq(problems(_LAT, batch_size='4'), ["batch_size: expected int, got '4'"])
test_eq(names(problems(_LAT, recipe=[0.3])), {'recipe'})
test_eq(names(problems(_LAT, variant={'prune': 0.3})), {'variant'})
test_eq(names(problems(_LAT, runtime_version=8.2)), {'runtime_version'})
test_eq(names(problems(_LAT, throttled='no')), {'throttled'})
test_eq(names(problems(_LAT, batch_size=4.0)), {'batch_size'})
test_eq(names(problems(_LAT, latency_ms_p50=float('inf'))), {'latency_ms_p50'})
test_eq(names(problems(_LAT, layer_precisions=[52, 9])), {'layer_precisions'})

# a bool is an int in Python, and neither a count nor a measure here
test_eq(problems(_LAT, batch_size=True), ['batch_size: expected int, got True'])
test_eq(names(problems(_LAT, schema_version=True)), {'schema_version'})
test_eq(names(problems(_LAT, latency_ms_p50=True)), {'latency_ms_p50'})
test_eq(names(problems(_LAT, engine_layers=True)), {'engine_layers'})
test_eq(names(problems(_QUAL, eval_n=False)), {'eval_n'})

# out of range
for _f, _v in [('batch_size', 0), ('repeats', 0), ('input_resolution', 0), ('latency_ms_p50', 0.0),
               ('latency_ms_p99', 0.0), ('size_mb', -1.0), ('latency_spread_pct', -0.1), ('warmup_s', -1.0),
               ('duration_s', -1.0), ('host_latency_ms', 0), ('engine_layers', 0), ('reformat_layers', -1)]:
    test_eq(names(problems(_LAT, **{_f: _v})), {_f})
test_eq(problems(_LAT, warmup_s=0, reformat_layers=0), [])
test_eq(names(problems(_QUAL, eval_n=0)), {'eval_n'})
test_eq(names(problems(_QUAL, argmax_parity=100.5)), {'argmax_parity'})
test_eq(names(problems(_QUAL, accuracy=150.0, ci_high=160.0)), {'accuracy', 'ci_high'})

# a count per precision
test_eq(problems(_LAT, layer_precisions={}), [])
test_eq(names(problems(_LAT, layer_precisions={'int8': -1})), {'layer_precisions'})
test_eq(names(problems(_LAT, layer_precisions={'int8': 2.5})), {'layer_precisions'})
test_eq(names(problems(_LAT, layer_precisions={'int8': True})), {'layer_precisions'})
test_eq(problems(_LAT, layer_precisions={'int8': np.int64(52)}), [])

# closed vocabularies
for _r, _f, _v in [(_LAT, 'precision', 'bf16'), (_LAT, 'weights', 'pretrained'), (_LAT, 'runtime', 'TensorRT 8.2.1'),
                   (_LAT, 'device_host', 'borrowed'), (_LAT, 'latency_source', 'guessed'),
                   (_QUAL, 'precision', 'bf16'), (_QUAL, 'ci_method', 'normal'), (_QUAL, 'measured_through', 'notebook')]:
    test_eq(names(problems(_r, **{_f: _v, **({'int8_mode': None} if _f == 'precision' and _r is _LAT else {})})), {_f})

# schema version mismatch
test_eq(problems(_LAT, schema_version=2), ['schema_version: 2 not one of (1,)'])

# unknown fields: a derived throughput, a verdict, and a name from before the split
test_eq(problems(_LAT, throughput_img_s=500.0), ['throughput_img_s: unknown field for a latency row'])
test_eq(problems(_QUAL, passed=True), ['passed: unknown field for a quality row'])
test_eq(problems(_LAT, latency_ms=8.0), ['latency_ms: unknown field for a latency row'])

In [ ]:
# int8_mode is required iff the precision is int8
test_eq(problems(_LAT, int8_mode=None), ['int8_mode: missing, required when precision is int8'])
test_eq(problems(_LAT, precision='fp16'), ["int8_mode: 'explicit' given for precision 'fp16'"])
test_eq(problems(_LAT, precision='mixed'), ["int8_mode: 'explicit' given for precision 'mixed'"])
test_eq(problems(_LAT, precision='fp16', int8_mode=None), [])
test_eq(names(problems(_LAT, int8_mode='dynamic')), {'int8_mode'})

# random weights come from no named model
test_eq(problems(_RAND, model_id='org/resnet50-p30'), ["model_id: 'org/resnet50-p30' given for random weights"])
test_eq(problems(_LAT, model_id=None, model_revision=None), [])   # trained weights need not name their model

# a quality row names the engine it was measured through, unless it was measured on the development model
test_eq(problems(_QUAL, engine_hash=None), ['engine_hash: missing, required when measured through the deployed engine'])
test_eq(problems(_QUAL, engine_hash=None, measured_through='dev_model'), [])

# ordered fields, among those recorded
test_eq(names(problems(_QUAL, accuracy=80.0)), {'ci_low, accuracy, ci_high'})
test_eq(names(problems(_QUAL, ci_low=79.0)), {'ci_low, accuracy, ci_high'})
test_eq(problems(_QUAL, ci_low=78.9, ci_high=78.9), [])
test_eq(names(problems(_LAT, latency_ms_p90=7.9)), {'latency_ms_p50, latency_ms_p90, latency_ms_p99'})
test_eq(names(problems(_LAT, latency_ms_p90=..., latency_ms_p99=7.0)), {'latency_ms_p50, latency_ms_p90, latency_ms_p99'})
test_eq(problems(_LAT, latency_ms_p90=8.0, latency_ms_p99=8.0), [])
test_eq(problems(_LAT, latency_ms_p90=None), [])
test_eq(names(problems(_LAT, reformat_layers=62)), {'reformat_layers, engine_layers'})

# digests are 64 lowercase hex characters
for _r, _f in [(_LAT, 'engine_hash'), (_LAT, 'graph_hash'), (_QUAL, 'engine_hash')]:
    for _bad in ('ab' * 31, 'AB' * 32, 'sha256:' + 'ab' * 32): test_eq(names(problems(_r, **{_f: _bad})), {_f})

# the date is ISO-8601 in UTC, and a real date
test_eq(problems(_LAT, measured_at='2026-09-28T10:00:00.123+00:00'), [])
test_eq(names(problems(_LAT, measured_at='2026-09-28')), {'measured_at'})
test_eq(names(problems(_QUAL, measured_at='2026-09-28T10:00:00+02:00')), {'measured_at'})
test_eq(names(problems(_LAT, measured_at='2026-13-28T10:00:00Z')), {'measured_at'})

# recipe keys are free, but the recipe is stored as JSON
test_eq(problems(_LAT, recipe={}), [])
test_eq(names(problems(_LAT, recipe={'x': object()})), {'recipe'})

In [ ]:
for _r in (_LAT, _RAND, _QUAL): test_eq(measurement.status(_r), ('VALID', []))


def reasons(row, **change):
    "measurement.status() reasons for `row` with some fields changed"
    v, rs = measurement.status(edit(row, **change))
    test_eq(v, 'INVALID')
    return rs


# latency: the device, not the weights
test_eq(reasons(_LAT, latency_source='estimated'), ['latency_source: latency not measured on the device'])
test_eq(reasons(_LAT, throttled=True), ['throttled: the device throttled during the session'])
test_eq(reasons(_LAT, throttled=None), ['throttled: not recorded'])
test_eq(reasons(_RAND, throttled=...), ['throttled: not recorded'])
test_eq(reasons(_LAT, latency_spread_pct=5.1), ['latency_spread_pct: 5.1 above 5.0'])
test_eq(reasons(_LAT, latency_spread_pct=None), ['latency_spread_pct: not recorded'])
test_eq(measurement.status(edit(_LAT, latency_spread_pct=5.0)), ('VALID', []))
test_eq(measurement.status(edit(_LAT, latency_spread_pct=5.1), max_spread_pct=6.0), ('VALID', []))
# the informational conditions do not decide validity
test_eq(measurement.status(edit(_LAT, temp_start_c=None, temp_end_c=None, gpu_freq_mhz=None, clocks_locked=None,
                                warmup_s=None, layer_precisions=None)), ('VALID', []))
test_eq(names(reasons(_LAT, throttled=None, latency_spread_pct=None, latency_source='estimated')),
        {'throttled', 'latency_spread_pct', 'latency_source'})

# quality: through the engine, with parity
test_eq(reasons(_QUAL, measured_through='dev_model'), ['measured_through: accuracy not measured through the deployed engine'])
test_eq(reasons(_QUAL, argmax_parity=98.9), ['argmax_parity: 98.9 below parity_floor 99.0'])
test_eq(reasons(_QUAL, argmax_parity=None), ['argmax_parity: not recorded'])
test_eq(measurement.status(edit(_QUAL, argmax_parity=99.0)), ('VALID', []))   # at the floor holds
test_eq(names(reasons(_QUAL, measured_through='dev_model', engine_hash=None, argmax_parity=...)),
        {'measured_through', 'argmax_parity'})

# a malformed row is invalid for its structural problems
test_eq(reasons(_LAT, batch_size=True), ['batch_size: expected int, got True'])
test_eq(reasons(_LAT, kind=...), ['kind: missing'])

In [ ]:
test_eq(measurement.key(_LAT)[0], 'latency')
test_eq(measurement.key(_QUAL)[0], 'quality')
test_eq(measurement.key(_LAT), measurement.key(dict(_LAT)))
# a latency row is unique on its graph and how it ran; a software update on the device is a new key
for _f, _v in [('graph_hash', '12' * 32), ('weights', 'random'), ('precision', 'fp16'), ('int8_mode', 'implicit'),
               ('batch_size', 1), ('device', 'raspberry-pi-5'), ('runtime', 'onnxruntime'),
               ('runtime_version', '8.5.2'), ('platform_version', 'jetpack-5.1.3'),
               ('measured_at', '2026-09-29T10:00:00Z')]:
    test_ne(measurement.key({**_LAT, _f: _v}), measurement.key(_LAT))
# the weights' values are not: another engine of the same graph and weights kind, the same key
test_eq(measurement.key({**_LAT, 'engine_hash': '12' * 32, 'model_id': None, 'run_id': 'l9', 'latency_ms_p50': 9.0,
                         'series': None, 'variant': 'renamed'}), measurement.key(_LAT))
# a quality row is unique on its model, engine, evaluation set, metric and date
for _f, _v in [('model_id', 'org/other'), ('model_revision', 'e4f5a6b'), ('engine_hash', '12' * 32),
               ('eval_set_hash', 'sha256:10'), ('metric', 'top5'), ('measured_at', '2026-09-29T10:00:00Z')]:
    test_ne(measurement.key({**_QUAL, _f: _v}), measurement.key(_QUAL))
test_eq(measurement.key({**_QUAL, 'accuracy': 70.0, 'run_id': 'q9', 'device': None}), measurement.key(_QUAL))

# throughput is derived from a latency row: batch_size * 1000 / latency_ms_p50
test_eq(measurement.throughput(_LAT), 500.0)
test_close(measurement.throughput({**_LAT, 'batch_size': 1, 'latency_ms_p50': 2.1}), 476.19, eps=0.01)
with ExceptionExpected(ValueError, 'latency row'): measurement.throughput(_QUAL)

In [ ]:
_j = measurement.join([_LAT, _RAND], [_QUAL])
test_eq(len(_j), 1)   # the random-weights engine has no accuracy to join, and needs none
_row = _j[0]
# identity kept per side, schema version once, the kind gone
test_eq((_row['latency_run_id'], _row['quality_run_id']), ('l1', 'q1'))
test_eq((_row['latency_measured_at'], _row['quality_measured_at']), (_LAT['measured_at'], _QUAL['measured_at']))
test_eq(_row['schema_version'], 1)
test_eq('kind' in _row or 'run_id' in _row, False)
# every field of both sides, nothing overwritten
_both = [f for f in measurement.FIELDS['latency'] if f in measurement.FIELDS['quality']]
_own = [f for f in _both if f not in ('schema_version', 'kind', 'engine_hash', 'precision', 'device', 'model_id',
                                      'model_revision')]
test_eq(len(_row), len(measurement.FIELDS['latency']) + len(measurement.FIELDS['quality']) - len(_both) + len(_own) - 1)
test_eq((_row['latency_ms_p50'], _row['accuracy'], _row['engine_hash']), (8.0, 78.9, _ENGINE))
test_eq(measurement.throughput({**_row, 'kind': 'latency'}), 500.0)

# only VALID rows of the right kind are paired, so the file read back can be given as both arguments
test_eq(len(measurement.join([_LAT, _RAND, _QUAL], [_LAT, _RAND, _QUAL])), 1)
test_eq(measurement.join([{**_LAT, 'throttled': True}], [_QUAL]), [])
test_eq(measurement.join([_LAT], [{**_QUAL, 'argmax_parity': 90.0}]), [])
test_eq(measurement.join([_LAT], [{**_QUAL, 'engine_hash': '12' * 32}]), [])
# one latency row, two quality rows of the same engine: two pairs
test_eq(len(measurement.join([_LAT], [_QUAL, {**_QUAL, 'run_id': 'q2', 'eval_set_hash': 'sha256:10'}])), 2)

# a field only one side recorded is taken from it; two sides that disagree on one engine raise
test_eq(measurement.join([{**_LAT, 'model_id': None, 'model_revision': None}], [_QUAL])[0]['model_id'], 'org/resnet50-p30')
test_eq(measurement.join([_LAT], [{**_QUAL, 'device': None}])[0]['device'], 'jetson-nano-4gb')
with ExceptionExpected(ValueError, 'precision'): measurement.join([_LAT], [{**_QUAL, 'precision': 'fp16'}])

In [ ]:
import sys, torch

# without onnx installed, a one-line error names the extra
_saved = sys.modules.get('onnx')
sys.modules['onnx'] = None
try:
    with ExceptionExpected(ImportError, r"fastermodels\[onnx\]"): measurement.graph_hash('x.onnx')
finally:
    if _saved is None: sys.modules.pop('onnx', None)
    else: sys.modules['onnx'] = _saved

import onnx


def export(width, seed, path):
    "A small conv net with `width` channels and weights drawn from `seed`, exported to ONNX"
    torch.manual_seed(seed)
    net = torch.nn.Sequential(torch.nn.Conv2d(3, width, 3), torch.nn.ReLU(), torch.nn.Flatten(),
                              torch.nn.Linear(width * 6 * 6, 10)).eval()
    torch.onnx.export(net, torch.randn(1, 3, 8, 8), str(path), dynamo=False, opset_version=17,
                      input_names=['input'], output_names=['output'])
    return path


with tempfile.TemporaryDirectory() as _d:
    _a, _b, _c = (export(w, s, Path(_d, f'{n}.onnx')) for n, w, s in (('a', 8, 0), ('b', 8, 1), ('c', 16, 0)))
    test_ne(Path(_a).read_bytes(), Path(_b).read_bytes())   # the files differ: the weights differ
    _h = measurement.graph_hash(_a)
    test_eq(bool(re.fullmatch(r'[0-9a-f]{64}', _h)), True)
    test_eq(measurement.graph_hash(str(_a)), _h)                 # deterministic, str or Path
    test_eq(measurement.graph_hash(_b), _h)                      # same shape, other weights: same graph
    test_ne(measurement.graph_hash(_c), _h)                      # another width: another graph
    # the file is read, not modified
    _before = Path(_a).read_bytes(); measurement.graph_hash(_a); test_eq(Path(_a).read_bytes(), _before)

    # initializer names are part of the graph
    _m = onnx.load(str(_a))
    _old = _m.graph.initializer[0].name
    _m.graph.initializer[0].name = 'renamed'
    for _n in _m.graph.node: _n.input[:] = ['renamed' if i == _old else i for i in _n.input]
    onnx.save(_m, str(Path(_d, 'r.onnx')))
    test_ne(measurement.graph_hash(Path(_d, 'r.onnx')), _h)

    # so is the element type: the same graph in fp16 is another graph
    _m = onnx.load(str(_a))
    _m.graph.initializer[0].data_type = onnx.TensorProto.FLOAT16
    onnx.save(_m, str(Path(_d, 'h.onnx')))
    test_ne(measurement.graph_hash(Path(_d, 'h.onnx')), _h)

    # weights stored outside the file are not read, and the hash does not depend on where they are stored
    _m = onnx.load(str(_b))
    onnx.save(_m, str(Path(_d, 'ext.onnx')), save_as_external_data=True, all_tensors_to_one_file=True,
              location='ext.data', size_threshold=0)
    test_eq(measurement.graph_hash(Path(_d, 'ext.onnx')), _h)

In [ ]:
with tempfile.TemporaryDirectory() as _d:
    _p = Path(_d, 'rows.jsonl')

    # rows of both kinds are appended to one file, and read back equal
    for _r in (_LAT, _RAND, _QUAL): measurement.write_row(_p, _r)
    test_eq(measurement.read_rows(_p), [_LAT, _RAND, _QUAL])
    test_eq(len(_p.read_text().splitlines()), 3)

    # an invalid row is refused, naming its problems, and nothing is written
    with ExceptionExpected(ValueError, 'batch_size'): measurement.write_row(_p, {**_LAT, 'batch_size': 0, 'run_id': 'l3'})
    with ExceptionExpected(ValueError, 'kind: missing'): measurement.write_row(_p, edit(_LAT, kind=...))
    # a duplicate key is refused, even with other fields changed
    with ExceptionExpected(ValueError, 'already in'): measurement.write_row(_p, {**_LAT, 'run_id': 'again', 'latency_ms_p50': 7.0})
    with ExceptionExpected(ValueError, 'already in'): measurement.write_row(_p, {**_QUAL, 'run_id': 'again', 'accuracy': 79.0})
    test_eq(measurement.read_rows(_p), [_LAT, _RAND, _QUAL])

    # uniqueness is per kind: a quality row never collides with a latency row
    _r4 = {**_LAT, 'run_id': 'l4', 'batch_size': 1}
    measurement.write_row(_p, _r4)

    # numpy scalars are written as plain JSON numbers
    _np = {**_LAT, 'run_id': 'l5', 'batch_size': np.int64(2), 'latency_ms_p50': np.float64(4.0),
           'recipe': {'prune': np.float64(0.3), 'calib_n': np.int64(512)}, 'layer_precisions': {'int8': np.int64(61)}}
    measurement.write_row(_p, _np)
    test_eq(json.loads(_p.read_text().splitlines()[-1])['batch_size'], 2)
    test_eq(measurement.read_rows(_p)[-1]['recipe'], {'prune': 0.3, 'calib_n': 512})

    # a row written by hand without a final newline does not swallow the next one
    _p.write_text(_p.read_text().rstrip('\n'))
    _r6 = {**_QUAL, 'run_id': 'q6', 'metric': 'top5'}
    measurement.write_row(_p, _r6)
    test_eq([r['run_id'] for r in measurement.read_rows(_p)], ['l1', 'l2', 'q1', 'l4', 'l5', 'q6'])

    # a malformed line is reported with its line number; blank lines are skipped
    _bad = Path(_d, 'bad.jsonl')
    _bad.write_text(json.dumps(_LAT) + '\n\n{"run_id": \n')
    with ExceptionExpected(ValueError, 'bad.jsonl:3'): measurement.read_rows(_bad)
    with ExceptionExpected(ValueError, 'bad.jsonl:3'): measurement.write_row(_bad, _QUAL)
    _bad.write_text('[1, 2]\n')
    with ExceptionExpected(ValueError, 'bad.jsonl:1: not a JSON object'): measurement.read_rows(_bad)

    # reading a file that is not there is an error, not an empty table
    with ExceptionExpected(FileNotFoundError): measurement.read_rows(Path(_d, 'none.jsonl'))

In [ ]:
import importlib.util

# without the extra installed, a one-line error names it (pandas blocked before any import of it)
_saved = sys.modules.get('pandas')
sys.modules['pandas'] = None
try:
    with ExceptionExpected(ImportError, r"fastermodels\[parquet\]"): measurement.to_parquet([_LAT], 'x.parquet')
finally:
    if _saved is None: sys.modules.pop('pandas', None)
    else: sys.modules['pandas'] = _saved

if importlib.util.find_spec('pyarrow') and importlib.util.find_spec('pandas'):
    import pandas as pd
    with tempfile.TemporaryDirectory() as _d:
        _pq = Path(_d, 'latency.parquet')
        _sparse = {k: v for k, v in _LAT.items() if k not in ('series', 'latency_ms_p99', 'layer_precisions')}
        measurement.to_parquet([_LAT, _RAND, {**_sparse, 'run_id': 'l3', 'batch_size': 1}], _pq)
        _df = pd.read_parquet(_pq)
        # one column per field of the kind, in schema order, one line per row
        test_eq(list(_df.columns), list(measurement.FIELDS['latency']))
        test_eq(len(_df), 3)
        # dict fields are canonical JSON: key order is not a difference
        test_eq(_df['recipe'][0], '{"calib_n":512,"prune":0.3,"quant":"int8"}')
        test_eq(json.loads(_df['layer_precisions'][0]), {'int8': 52, 'fp16': 9})
        test_eq(_df['weights'].tolist(), ['trained', 'random', 'trained'])
        test_eq(_df['model_id'].tolist(), ['org/resnet50-p30', None, 'org/resnet50-p30'])
        # an absent field is a null, not a hole
        test_eq(_df['series'].isna().tolist(), [False, False, True])
        test_eq(_df['layer_precisions'].isna().tolist(), [False, False, True])

        measurement.to_parquet([_QUAL], Path(_d, 'quality.parquet'))
        test_eq(list(pd.read_parquet(Path(_d, 'quality.parquet')).columns), list(measurement.FIELDS['quality']))

        # one kind per file; no rows, no file; a malformed row is refused with its index; nothing is written
        with ExceptionExpected(ValueError, 'one kind per file'): measurement.to_parquet([_LAT, _QUAL], Path(_d, 'no.parquet'))
        with ExceptionExpected(ValueError, 'no rows'): measurement.to_parquet([], Path(_d, 'no.parquet'))
        with ExceptionExpected(ValueError, 'row 1 refused'): measurement.to_parquet([_LAT, {**_LAT, 'batch_size': 0}], Path(_d, 'no.parquet'))
        test_eq(Path(_d, 'no.parquet').exists(), False)
else: print('pandas or pyarrow not installed: to_parquet round trip skipped')